# 🧬 Phase 1: Biomedical KG — Data Download & RAPIDS GPU Preprocessing

**Project:** BioKG Disease Link Predictor  
**Dataset:** DRKG (Drug Repurposing Knowledge Graph) — 5.8M biological triples  
**Tech:** NVIDIA RAPIDS cuDF/cuGraph + DRKG exploration  

---

## 🎓 What You'll Learn in This Notebook

1. **What a Knowledge Graph is** — entities, relations, triples
2. **NVIDIA RAPIDS cuDF** — GPU-accelerated DataFrames (10-100x faster than pandas)
3. **cuGraph** — GPU graph analytics algorithms (PageRank, degree statistics)
4. **DRKG structure** — the real biomedical dataset used in COVID-19 drug repurposing

## ⚡ Before Starting
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Click **Connect** (top right)
3. Run all cells in order (Shift+Enter)

## Step 0: Check GPU

In [ ]:
# Let's check what GPU Colab gave us
!nvidia-smi

import torch
print(f'\n✅ CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')

## Step 1: Install NVIDIA RAPIDS

**🎓 What is RAPIDS?**

NVIDIA RAPIDS is a suite of GPU-accelerated data science libraries:
- **cuDF**: GPU DataFrames — same API as pandas, but runs on GPU
- **cuGraph**: GPU Graph Analytics — same concepts as NetworkX, but 100x faster
- **cuML**: GPU Machine Learning — same as scikit-learn, but on GPU

**Why does this matter?**
When Gurdeep said 'experience applying DL methods on large datasets on GPU clusters', he means exactly this — not just training neural networks, but the ENTIRE pipeline: data loading, preprocessing, graph analytics — all on GPU.

This cell will take ~3-5 minutes. ☕

In [ ]:
import subprocess
import sys

# Detect CUDA version for correct RAPIDS install
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_output = result.stdout + result.stderr
print('CUDA version info:', cuda_output[:200])

# Install RAPIDS cuDF and cuGraph (matching Colab's CUDA version)
print('\n📦 Installing NVIDIA RAPIDS (this takes 3-5 minutes)...')
!pip install cudf-cu12 cugraph-cu12 --extra-index-url=https://pypi.nvidia.com -q

print('\n✅ RAPIDS installed!')

In [ ]:
# Install other dependencies
!pip install loguru rich tqdm requests -q

# Verify RAPIDS works
import cudf
import cugraph
print(f'✅ cuDF version: {cudf.__version__}')
print(f'✅ cuGraph version: {cugraph.__version__}')

# Quick test: create a GPU DataFrame
test_df = cudf.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
print(f'\n🧪 cuDF test: {test_df.sum().to_dict()}')
print('cuDF is working on GPU! 🎉')

## Step 2: Download DRKG Dataset

**🎓 What is DRKG?**

The **Drug Repurposing Knowledge Graph** is a comprehensive biological knowledge graph from Microsoft Research, used in COVID-19 drug repurposing research.

It integrates data from 7 biological databases:
| Database | Content |
|----------|--------|
| **Hetionet** | Gene-disease, gene-compound relationships |
| **GNBR** | Gene-disease-compound associations from literature |
| **STRING** | Protein-protein interactions |
| **DrugBank** | Drug targets, mechanisms |
| **DGIdb** | Drug-gene interactions |
| **DGIDB** | Drug-gene interactions |
| **Intact** | Molecular interaction data |

**What is a triple?**
- A triple is: **(head_entity, relation, tail_entity)**
- Example: `(Gene::BRCA1, hetionet::GaD::Gene:Disease, Disease::Breast_Cancer)`
- This means: 'Gene BRCA1 is associated with Breast Cancer'
- The entire KG is a collection of such factual statements stored as triples

In [ ]:
import requests
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def download_file(url, dest_path, desc='Downloading'):
    """Download a file with a progress bar."""
    if dest_path.exists():
        print(f'⏭️  {dest_path.name} already exists, skipping.')
        return
    
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total = int(response.headers.get('content-length', 0))
    
    with open(dest_path, 'wb') as f, tqdm(desc=desc, total=total, unit='iB', unit_scale=True) as pb:
        for chunk in response.iter_content(8192):
            pb.update(f.write(chunk))
    
    print(f'✅ Saved: {dest_path.name} ({dest_path.stat().st_size/1024**2:.1f} MB)')

# Download DRKG
# Note: This is the compressed version (~200MB compressed, ~800MB uncompressed)
print('📥 Downloading DRKG dataset...')
print('   This is a REAL biomedical dataset used in COVID-19 drug research!')
print('   Source: Microsoft Research GitHub\n')

download_file(
    'https://dgl-data.s3-accelerate.amazonaws.com/dataset/DRKG/drkg.tar.gz',
    DATA_DIR / 'drkg.tar.gz',
    desc='DRKG triples'
)

# Extract
import tarfile
print('\n📦 Extracting...')
with tarfile.open(DATA_DIR / 'drkg.tar.gz', 'r:gz') as tar:
    tar.extractall(DATA_DIR)

# Find the TSV file
tsv_files = list(DATA_DIR.glob('**/*.tsv'))
print(f'\n📂 Found TSV files: {[f.name for f in tsv_files]}')

DRKG_TSV = tsv_files[0] if tsv_files else DATA_DIR / 'drkg.tsv'
print(f'\n✅ DRKG file: {DRKG_TSV}')
print(f'   Size: {DRKG_TSV.stat().st_size / 1024**2:.1f} MB')

## Step 3: Load with cuDF (GPU DataFrame)

**🎓 The Speed Comparison**

This is where RAPIDS shines. We'll load 5.8M rows:
- **pandas (CPU)**: ~15-30 seconds
- **cuDF (GPU)**: ~1-3 seconds

The key insight: cuDF loads data DIRECTLY into GPU VRAM, with no CPU bottleneck. For large-scale ML pipelines, this data loading speedup alone can save hours per day.

In [ ]:
import time
import cudf
import pandas as pd

print('=' * 60)
print('⚡ BENCHMARK: cuDF (GPU) vs pandas (CPU)')
print('=' * 60)

# ── CPU Benchmark ──────────────────────────────────────────────────────────
print('\n🐢 Loading with pandas (CPU)...')
t0 = time.time()
df_pandas = pd.read_csv(str(DRKG_TSV), sep='\t', header=None, names=['head', 'relation', 'tail'])
cpu_time = time.time() - t0
print(f'   Time: {cpu_time:.2f}s | Rows: {len(df_pandas):,}')

# ── GPU Benchmark ──────────────────────────────────────────────────────────
print('\n⚡ Loading with cuDF (GPU)...')
t0 = time.time()
df = cudf.read_csv(str(DRKG_TSV), sep='\t', header=None, names=['head', 'relation', 'tail'])
gpu_time = time.time() - t0
print(f'   Time: {gpu_time:.2f}s | Rows: {len(df):,}')

# ── Results ────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
speedup = cpu_time / gpu_time
print(f'🚀 cuDF is {speedup:.1f}x faster than pandas!')
print(f'   CPU: {cpu_time:.2f}s → GPU: {gpu_time:.2f}s')
print('=' * 60)

# Store speedup for the LinkedIn post
RAPIDS_SPEEDUP = speedup

# Preview the data
print('\n📊 First 5 rows:')
print(df.head().to_pandas().to_string(index=False))

## Step 4: Explore the Knowledge Graph

Let's understand what's in DRKG before we train on it. Good ML engineers always understand their data first!

In [ ]:
print('📊 DRKG Knowledge Graph Statistics')
print('=' * 50)
print(f'Total triples: {len(df):,}')

# ── Entity Type Distribution ───────────────────────────────────────────────
# DRKG entities follow the format: 'TypeName::EntityID'
# We extract the type prefix
print('\n🔍 Extracting entity types...')
print('   (cuDF string operations run on GPU!)')

# GPU string split operation
head_types = df['head'].str.split('::').list.get(0)
tail_types = df['tail'].str.split('::').list.get(0)
relation_sources = df['relation'].str.split('::').list.get(0)

# Count occurrences (all on GPU)
head_type_counts = head_types.value_counts().to_pandas()
tail_type_counts = tail_types.value_counts().to_pandas()
relation_source_counts = relation_sources.value_counts().to_pandas()

# Merge head and tail type counts
all_type_counts = head_type_counts.add(tail_type_counts, fill_value=0).sort_values(ascending=False)

print('\n📊 Entity Types in DRKG:')
print('-' * 40)
total_entities = all_type_counts.sum()
for entity_type, count in all_type_counts.head(15).items():
    bar = '█' * int(30 * count / all_type_counts.max())
    print(f'{entity_type:<30} {int(count):>10,}  {bar}')

print('\n📊 Relation Sources (which databases contributed):')
print('-' * 40)
for source, count in relation_source_counts.items():
    bar = '█' * int(30 * count / relation_source_counts.max())
    print(f'{source:<20} {count:>10,}  {bar}')

In [ ]:
import matplotlib.pyplot as plt

# Visualize entity type distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('DRKG Knowledge Graph Analysis\n(5.8M biological triples, GPU-analyzed with NVIDIA RAPIDS)', 
             fontsize=13, fontweight='bold')

# Entity types
ax1 = axes[0]
colors = plt.cm.viridis([i/len(all_type_counts) for i in range(len(all_type_counts))])
bars = ax1.barh(all_type_counts.index[:10][::-1], all_type_counts.values[:10][::-1], color=colors[:10][::-1])
ax1.set_xlabel('Count', fontsize=11)
ax1.set_title('Top 10 Entity Types', fontsize=12, fontweight='bold')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
for bar, val in zip(bars, all_type_counts.values[:10][::-1]):
    ax1.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2, 
             f'{int(val):,}', va='center', fontsize=9)

# Relation sources (pie chart)
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    relation_source_counts.values,
    labels=relation_source_counts.index,
    autopct='%1.1f%%',
    colors=plt.cm.Set3(range(len(relation_source_counts)))
)
ax2.set_title('Relation Sources (Databases)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('drkg_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Plot saved as drkg_analysis.png')
print('   → Use this in your GitHub README and LinkedIn post!')

## Step 5: cuGraph — GPU Graph Analytics

**🎓 What is cuGraph?**

cuGraph is NVIDIA's GPU-accelerated graph analytics library. It implements algorithms like:
- **PageRank**: Identify the most 'central' nodes in the graph
- **Betweenness Centrality**: Nodes that act as bridges
- **Connected Components**: Find isolated subgraphs
- **Triangle Counting**: Graph clustering

**Why PageRank on diseases?**
A disease with high PageRank is connected to many important genes, drugs, and pathways — it's a 'hub' disease. These are often the most studied (and well-represented in our training data), which affects model performance.

In [ ]:
import cugraph

print('🔗 Building Knowledge Graph with cuGraph...')
print('   (Building a graph from 5.8M edges on GPU!)')

# We need integer node IDs for cuGraph
# Build entity mapping
all_entities = cudf.concat([df['head'], df['tail']]).unique().reset_index(drop=True)
entity_to_id = cudf.Series(range(len(all_entities)), index=all_entities)

print(f'\n📊 Graph info:')
print(f'   Unique entities (nodes): {len(all_entities):,}')
print(f'   Triples (edges): {len(df):,}')

# Map string entities to integers
t0 = time.time()
edges = cudf.DataFrame({
    'src': df['head'].map(entity_to_id),
    'dst': df['tail'].map(entity_to_id),
}).dropna().astype({'src': 'int32', 'dst': 'int32'})

# Build cuGraph Graph
G = cugraph.Graph(directed=True)
G.from_cudf_edgelist(edges, source='src', destination='dst')
build_time = time.time() - t0

print(f'\n✅ Graph built in {build_time:.2f}s')
print(f'   Nodes: {G.number_of_nodes():,}')
print(f'   Edges: {G.number_of_edges():,}')

# Compute PageRank on GPU
print('\n⚡ Computing PageRank on GPU...')
t0 = time.time()
pagerank_df = cugraph.pagerank(G, alpha=0.85, max_iter=100)
pr_time = time.time() - t0
print(f'✅ PageRank computed in {pr_time:.2f}s for {G.number_of_nodes():,} nodes!')
print(f'   On CPU (NetworkX), this would take minutes!')

In [ ]:
# Find top PageRank entities
id_to_entity = {int(v): k for k, v in entity_to_id.to_pandas().items()}

top_pr = pagerank_df.sort_values('pagerank', ascending=False).head(20).to_pandas()
top_pr['entity'] = top_pr['vertex'].map(id_to_entity)
top_pr['type'] = top_pr['entity'].apply(lambda x: x.split('::')[0] if '::' in str(x) else 'Unknown')

print('\n🏆 Top 20 Most Central Entities (by PageRank):')
print('-' * 70)
print(f'{"Rank":<6} {"Type":<20} {"Entity":<40} {"PageRank"}')
print('-' * 70)
for i, row in top_pr.iterrows():
    entity_name = str(row['entity'])[:40] if row['entity'] else f"ID:{row['vertex']}"
    print(f"{i+1:<6} {row['type']:<20} {entity_name:<40} {row['pagerank']:.6f}")

# Separate disease nodes
print('\n🦠 Top Disease Hubs (most connected diseases):')
print('-' * 70)
disease_mask = top_pr['type'] == 'Disease'
disease_nodes = pagerank_df.to_pandas()
disease_nodes['entity'] = disease_nodes['vertex'].map(id_to_entity)
disease_nodes['type'] = disease_nodes['entity'].apply(lambda x: x.split('::')[0] if '::' in str(x) else '')
disease_hubs = disease_nodes[disease_nodes['type'] == 'Disease'].sort_values('pagerank', ascending=False).head(10)

for i, (_, row) in enumerate(disease_hubs.iterrows(), 1):
    entity_name = str(row['entity'])[:50]
    print(f'  {i:2d}. {entity_name:<50} PR={row["pagerank"]:.6f}')

## Step 6: Preprocess for Training

We need to:
1. Build **entity2id** and **relation2id** mappings (names → integers)
2. Split into **train/valid/test** (80/10/10)
3. Save as processed TSV files for the training script

In [ ]:
import json
import numpy as np

PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('🗺️  Building entity and relation ID mappings...')

# Get unique entities and relations (on GPU)
unique_entities = cudf.concat([df['head'], df['tail']]).unique().to_pandas()
unique_relations = df['relation'].unique().to_pandas()

# Build mappings
entity2id = {entity: int(idx) for idx, entity in enumerate(sorted(unique_entities))}
relation2id = {rel: int(idx) for idx, rel in enumerate(sorted(unique_relations))}

print(f'✅ Entities: {len(entity2id):,}')
print(f'✅ Relations: {len(relation2id):,}')

# Save mappings
with open(PROCESSED_DIR / 'entity2id.json', 'w') as f:
    json.dump(entity2id, f)
with open(PROCESSED_DIR / 'relation2id.json', 'w') as f:
    json.dump(relation2id, f)
print('✅ Mappings saved!')

# Train/val/test split
print('\n✂️  Splitting dataset...')
df_pandas = df.to_pandas()  # Need pandas for sample()
df_shuffled = df_pandas.sample(frac=1, random_state=42).reset_index(drop=True)

n = len(df_shuffled)
n_train = int(n * 0.8)
n_valid = int(n * 0.1)

df_train = df_shuffled.iloc[:n_train]
df_valid = df_shuffled.iloc[n_train:n_train + n_valid]
df_test = df_shuffled.iloc[n_train + n_valid:]

print(f'   Train: {len(df_train):,} triples')
print(f'   Valid: {len(df_valid):,} triples')
print(f'   Test:  {len(df_test):,} triples')

# Save splits
df_train.to_csv(PROCESSED_DIR / 'train.tsv', sep='\t', index=False, header=False)
df_valid.to_csv(PROCESSED_DIR / 'valid.tsv', sep='\t', index=False, header=False)
df_test.to_csv(PROCESSED_DIR / 'test.tsv', sep='\t', index=False, header=False)

# Save stats
stats = {
    'n_entities': len(entity2id),
    'n_relations': len(relation2id),
    'n_train': len(df_train),
    'n_valid': len(df_valid),
    'n_test': len(df_test),
    'rapids_speedup': round(RAPIDS_SPEEDUP, 2),
}
with open(PROCESSED_DIR / 'stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'\n✅ All files saved to {PROCESSED_DIR}/')
for f in PROCESSED_DIR.iterdir():
    print(f'   {f.name}: {f.stat().st_size/1024:.1f} KB')

## Step 7: Save Everything to Google Drive

Mount Google Drive to save your processed data — so you don't have to redownload next time!

In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Create project folder in Drive
DRIVE_DIR = Path('/content/drive/MyDrive/BioKG_Project')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy processed data
DRIVE_PROCESSED = DRIVE_DIR / 'data' / 'processed'
DRIVE_PROCESSED.mkdir(parents=True, exist_ok=True)

print('💾 Saving processed data to Google Drive...')
for f in PROCESSED_DIR.iterdir():
    dest = DRIVE_PROCESSED / f.name
    shutil.copy2(f, dest)
    print(f'   ✅ {f.name} → Drive')

# Save the analysis plot too
if Path('drkg_analysis.png').exists():
    shutil.copy2('drkg_analysis.png', DRIVE_DIR / 'drkg_analysis.png')
    print('   ✅ drkg_analysis.png → Drive')

print(f'\n🎉 Everything saved to Google Drive: {DRIVE_DIR}')
print('   Your data will persist across Colab sessions!')

## ✅ Phase 1 Complete! Summary

Here's what you accomplished in this notebook:

In [ ]:
print('=' * 60)
print('🎉 PHASE 1 COMPLETE!')
print('=' * 60)

with open(PROCESSED_DIR / 'stats.json') as f:
    stats = json.load(f)

print(f'''
📊 What you processed:
   • {stats["n_train"]:,} training triples
   • {stats["n_valid"]:,} validation triples  
   • {stats["n_test"]:,} test triples
   • {stats["n_entities"]:,} unique biomedical entities
   • {stats["n_relations"]:,} unique relation types

⚡ RAPIDS Performance:
   • cuDF was {stats["rapids_speedup"]}x faster than pandas!
   • PageRank computed on {stats["n_entities"]:,} nodes in seconds

🧠 What you learned:
   ✅ Knowledge graph triples (head, relation, tail)
   ✅ NVIDIA RAPIDS cuDF — GPU DataFrames
   ✅ NVIDIA RAPIDS cuGraph — GPU graph analytics
   ✅ PageRank algorithm on real biomedical data
   ✅ Train/val/test splitting for graph ML

🚀 Next Steps:
   → Phase 2: Encode entities with BioBridge/BiomedBERT
   → Phase 3: Train RotatE KGE model with PyTorch Lightning
   → Phase 4: Deploy FastAPI inference server
   → Phase 5: Launch Gradio demo on Hugging Face Spaces!
''')

print('📸 LinkedIn Post Material:')
print(f'   \'Just processed 5.8M biomedical triples from DRKG using')
print(f'   NVIDIA RAPIDS cuDF — {stats["rapids_speedup"]}x faster than pandas!')
print(f'   Building a GPU-accelerated disease link prediction system...\'') 